<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
Código suplementar do livro <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a>, de <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>Repositório de código: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

<!-- aviso-traducao-ptbr -->
<sub>
<b>Tradução não oficial para português do Brasil.</b> Este arquivo é uma obra
derivada do repositório original de Sebastian Raschka
(<a href="https://github.com/rasbt/reasoning-from-scratch">rasbt/reasoning-from-scratch</a>),
licenciado sob Apache License 2.0. Apenas o texto foi traduzido; o código
permanece inalterado. Não é uma publicação oficial da Manning e não substitui o
livro. Detalhes das convenções em <code>GLOSSARIO-TRADUCAO.md</code>.
</sub>

# Capítulo 6: Soluções dos exercícios

Pacotes usados neste notebook:

In [ ]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

&nbsp;
## Exercício 6.1: Adicionando reward shaping sensível ao formato

- Podemos atribuir um reward parcial (score 0.5) caso nenhuma resposta em "\boxed{}" seja encontrada, da seguinte forma, usando o fallback `fallback="number_then_full"` que programamos no capítulo 3:

In [ ]:
from reasoning_from_scratch.ch03 import (
    extract_final_candidate, grade_answer
)

def partial_reward_rlvr(answer_text, ground_truth):
    
    # 1) Try to extract a boxed answer
    boxed = extract_final_candidate(
        answer_text, fallback=None
    )
    if boxed:
        correct = grade_answer(boxed, ground_truth)
        return 1.0 if correct else 0.0

    # 2) If no boxed answer is found, look for number
    unboxed = extract_final_candidate(
        answer_text, fallback="number_then_full"
    )
    if unboxed:
        correct = grade_answer(unboxed, ground_truth)
        return 0.5 if correct else 0.0

    return 0.0

- Quando encaixada no código do capítulo 6 e treinada sob as mesmas configurações, a variante de reward parcial atinge acurácia menor (37,8%) que a configuração padrão de GRPO (47,4%), apesar de usar um número parecido de tokens em média

| # | Método                                   | Step | Máx. de tokens | Nº de rollouts | Acurácia | Média de tokens |
|---|------------------------------------------|------|------------|--------------|----------|----------------|
| 1 | GRPO (capítulo 6)                        | 50   | 512        | 8            | 47,4%    | 586,11         |
| 2 | GRPO com rewards parciais (exercício 6.1) | 50   | 512        | 8            | 37,8%    | 550,33         |

&nbsp;
## Exercício 6.2: Casos de advantage zero

- Se os rewards forem todos iguais (por exemplo, todos 0 ou todos 1), os advantages serão todos 0, porque subtrair a média remove o valor de reward compartilhado e deixa apenas zeros, o que podemos demonstrar abaixo

In [3]:
import torch

rollout_rewards = [0., 0., 0., 0.]
rewards = torch.tensor(rollout_rewards)
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([0., 0., 0., 0.])


In [4]:
rollout_rewards = [1., 1., 1., 1.]
rewards = torch.tensor(rollout_rewards)
advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-4)

print(advantages)

tensor([0., 0., 0., 0.])


- Agora, se todos os advantages são 0, a loss também será zero, porque a loss multiplica os advantages pelas log probabilities, e multiplicar por zero elimina a contribuição

```python
pg_loss = -(advantages.detach() * logps).mean()
```

- Como resultado, o policy gradient é zero e os parâmetros do modelo não são atualizados para aquele prompt

- Esse comportamento é intencional; se todos os rollouts são igualmente ruins ou igualmente bons, não há sinal relativo que diga ao modelo qual comportamento reforçar ou suprimir
- Intuitivamente, se o modelo responde todas as perguntas corretamente, não há necessidade de atualizá-lo
- Da mesma forma, se o modelo responde todas as perguntas incorretamente, não queremos atualizar o modelo para reforçar esse comportamento